## Self Attention

Self-Attention'ın temelinde üç dönüşüm var:

* Query (Q): Bu konum hangi bilgiyi arıyor?

* Key (K): Her konum hangi bilgiyi temsil ediyor?

* Value (V): O konumdan aktarılacak bilgi ne?

In [3]:
import torch
import torch.nn as nn

In [4]:

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.query = nn.Conv2d(channels, channels // 8, 1)
        self.key = nn.Conv2d(channels, channels // 8, 1)
        self.value = nn.Conv2d(channels, channels, 1)

        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        d = C // 8

        q = self.query(x).flatten(2).transpose(1, 2)
        k = self.key(x).flatten(2)
        v = self.value(x).flatten(2).transpose(1, 2)

        attention = torch.bmm(q, k) * (d ** -0.5)
        attention = F.softmax(attention, dim=-1)

        out = torch.bmm(attention, v)
        out = out.transpose(1, 2).reshape(B, C, H, W)

        return self.gamma * out

## Multi-Head Attention (MHA)

Self-Attention'da tek bir attention işlemi yapıyorduk. Multi-Head Attention'da ise birden fazla attention başlığı paralel olarak çalışır.

Örneğin farklı başlıklar farklı ilişkileri öğrenebilir:

* Bir başlık yakın konumlar arasındaki ilişkileri,

* Bir diğeri uzak konumlar arasındaki ilişkileri,

*  Bir başkası farklı özellik örüntülerini öğrenebilir.

Bunlar olası davranışlardır; her başlığın mutlaka belirli bir görevi öğrenmesi garanti değildir.

nn.MultiheadAttention kritik ayrıntılar:

* batch_first=True: Giriş sırası (B, N, C).

* embed_dim=64: Her token'ın özellik boyutu.

* num_heads=8: Sekiz paralel attention başlığı.

* query=key=value=tokens: Self-Attention yapıyoruz.

* Çıkışın kanal ve uzamsal boyutları girişle aynı kalır.

In [6]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, channels, num_heads=8):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):
        B, C, H, W = x.shape

        # (B, C, H, W) -> (B, H*W, C)
        tokens = x.flatten(2).transpose(1, 2)

        # Self-attention
        out, _ = self.attention(
            tokens, tokens, tokens,
            need_weights=False
        )

        # (B, H*W, C) -> (B, C, H, W)
        out = out.transpose(1, 2).reshape(B, C, H, W)

        return out

In [7]:
x = torch.randn(8, 64, 14, 14)

mha = MultiHeadSelfAttention(
    channels=64,
    num_heads=8
)

y = mha(x)

print(y.shape)

torch.Size([8, 64, 14, 14])


In [9]:
x = torch.randn(8, 196, 64)

block = TransformerEncoderBlock(
    embed_dim=64,
    num_heads=8
)

y = block(x)

print(y.shape)

torch.Size([8, 196, 64])
